In [66]:
from astropy.io import fits
from astropy.table import Table
from astropy.table import vstack
import matplotlib.pyplot as plt
import numpy as np
import os

from pysersic.results import plot_image
from pysersic import check_input_data


FILTERS =["F606W", "F814W","F115W", "F150W", "F200W", "F277W", "F356W", "F444W"]

with fits.open("/nvme/scratch/work/alberttg/Summer_project/Ha_and_NII_broad_line_data.fits") as hdul:
        data = hdul[1].data
TABLE = Table(data)


In [67]:
def make_mask(data, id, filt):
    """
    Makes a mask for a cutout.

    Parameters
    ----------
        data : array-like
            data will be the segmentation info
        path : string
            path to my cutout folder for galaxy of given id
    Returns
    -------
        newhdu : array-like
            mask data
    """
    size = 32 # Automate this later

    x0 = int((size/2) - 1)
    y0 = x0
    data[data == data[x0,y0]] = 0  #change object to 0
    data[data > 0 ] = 1 #change other objects to 1
    newhdu=fits.PrimaryHDU(data)
    newhdu.writeto(f"/nvme/scratch/work/alberttg/Summer_project/Cutouts/{id}/{filt}_mask.fits", overwrite=True)
    # print(f'{ID} final mask saved as "{ID}mask_final.fits"')
        

In [68]:
def load_data(cutout_path, psf_path, id, filt):
    """
    Parameters
    ----------
        cutout_path : str

        psf_path : str

    Returns
    -------
        im : array-lie
            Science image data

        sig : array-like
            Weighted error map

        mask : array-like
            Mask made from segmentation data

        psf : array-like
            Normalised psf data
    """
    cutout = fits.open(cutout_path)

    im = cutout[1].data # science image data
    seg = cutout[2].data # Segmentation map
    sig = cutout[3].data # rms error map of uncertainties
    make_mask(seg, id, filt)
    mask = fits.open(f"/nvme/scratch/work/alberttg/Summer_project/Cutouts/{id}/{filt}_mask.fits")[0].data
    # mask.info()

    psf = fits.open(psf_path)[0].data # psf data

    # Cropping the psf because its 4as but I need it to be 0.96as to match my cutouts
    if len(psf) > len(im):
        # Crop centered
        diff = len(psf) - len(im)
        start = diff // 2
        end = start + len(im)
        psf = psf[start:end, start:end]
            
    # Normalise psf data 
    psf = psf / np.sum(psf) 

    # check_input_data(data=im, rms=sig, psf=psf, mask=mask) # Returns True if all checks pass, otherwise raises warning.

    # fig, ax = plot_image(im,mask,sig,psf)
             
    return im, sig, mask, psf




In [69]:
def run_for_all_galaxies(table):
    """
    """
    ids = table["SURVEY_ID"]

    surveys = table["SURVEY"]

    for i in range(len(ids)):
        for filt in FILTERS:

            cutout_path = f"/nvme/scratch/work/alberttg/Summer_project/Cutouts/{ids[i]}/{filt}.fits"

            psf_path = f"/nvme/scratch/work/alberttg/Summer_project/PSFs/{surveys[i]}/{filt}_psf_norm.fits"

            load_data(cutout_path, psf_path, ids[i], filt)


       


In [70]:
run_for_all_galaxies(TABLE)

In [71]:
import subprocess

# subprocess.run(f"python /nvme/scratch/work/westcottl/Codes/Morfometryka/Code/morfometryka965.py {science_path} {psf_path} noshow", shell=True)